In [1]:
import pandas as pd
import numpy as np
import ast

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

Burada:

pandas → veri setlerini okumak için,

numpy → sayısal işlemler için,

ast → JSON benzeri metinleri listeye çevirmek için,

CountVectorizer → metinleri sayısal vektöre çevirmek için,

cosine_similarity → filmler arasındaki benzerliği hesaplamak için

kullanılıyor.

**Drive'a Bağlanma**

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Google Drive dosya yollarını tanımlama**

In [22]:
movies_path = "/content/drive/MyDrive/ML Proje/Movie Recommendation System/tmdb_5000_movies.csv"
credits_path = "/content/drive/MyDrive/ML Proje/Movie Recommendation System/tmdb_5000_credits.csv"


**Veri setlerini okuma**

In [25]:
movies = pd.read_csv(movies_path)
credits = pd.read_csv(credits_path)

print("Movies shape:", movies.shape)
print("Credits shape:", credits.shape)

Movies shape: (4803, 20)
Credits shape: (4803, 4)


**Veri setlerini birleştirme**

In [26]:
movies = movies.merge(credits, on="title")
movies.shape

(4809, 23)

İki tabloyu title sütununa göre birleştiriyoruz.
Böylece film bilgileri ile oyuncu/ekip bilgileri tek tabloda birleşiyor.

**Gerekli sütunları seçme**

In [27]:
movies = movies[[
    "movie_id",
    "title",
    "overview",
    "genres",
    "keywords",
    "cast",
    "crew"
]]


Recommendation sistemi için tüm sütunlara ihtiyacımız yok.
Sadece şu bilgileri kullanıyoruz:

movie_id

title

overview

genres

keywords

cast

crew

In [28]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


**Eksik Verileri Kontrol Etme**

In [29]:
movies.isnull().sum()

,0
movie_id,0
title,0
overview,3
genres,0
keywords,0
cast,0
crew,0


**Eksik verileri güvenli şekilde temizleme**

In [30]:
movies.dropna(subset=[
    "overview",
    "genres",
    "keywords",
    "cast",
    "crew"
], inplace=True)

Burada sadece recommendation için gerekli sütunlarda boş olan satırları siliyoruz.

Bu en doğru yöntemdir.
Çünkü örneğin overview veya cast boşsa daha sonra hata alabiliriz.

**Tekrar Kontrol**

In [31]:
movies.isnull().sum()
print("After dropna shape:", movies.shape)

After dropna shape: (4806, 7)


**genres ve keywords sütunlarını parse etme**

In [32]:
def convert(text):
    result = []
    for item in ast.literal_eval(text):
        result.append(item["name"])
    return result

In [33]:
movies["genres"] = movies["genres"].apply(convert)
movies["keywords"] = movies["keywords"].apply(convert)

Örneğin:

Önce:

[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}]

Sonra:

["Action", "Adventure"]

**Oyuncu listesinden ilk 3 oyuncuyu alma**

In [34]:
def get_top_cast(text):
    result = []
    counter = 0

    for item in ast.literal_eval(text):
        if counter < 3:
            result.append(item["name"])
            counter += 1
        else:
            break

    return result

Şimdi Uygulayalım:

In [35]:
movies["cast"] = movies["cast"].apply(get_top_cast)

Tüm oyuncuları almak yerine ilk 3 oyuncuyu almak yeterlidir.
Bu hem sistemi sadeleştirir hem de en önemli oyuncuları korur.

**crew sütunundan yönetmeni alma**

In [36]:
def fetch_director(text):
    result = []

    for item in ast.literal_eval(text):
        if item["job"] == "Director":
            result.append(item["name"])
            break

    return result

Şimdi Uygulayalım:

In [37]:
movies["crew"] = movies["crew"].apply(fetch_director)

crew içinde birçok kişi vardır ama bizim için en önemli kişi yönetmendir.
Bu yüzden sadece Director bilgisi alınıyor.

**overview sütununu kelime listesine çevirme**

In [38]:
movies["overview"] = movies["overview"].apply(lambda x: x.split())

Örneğin:

"A thief enters dreams to steal secrets"

şuna dönüşür:

["A", "thief", "enters", "dreams", "to", "steal", "secrets"]

**Çok Kelimeli İsimlerde Boşluk Kaldırma**

In [39]:
movies["genres"] = movies["genres"].apply(lambda x: [i.replace(" ", "") for i in x])
movies["keywords"] = movies["keywords"].apply(lambda x: [i.replace(" ", "") for i in x])
movies["cast"] = movies["cast"].apply(lambda x: [i.replace(" ", "") for i in x])
movies["crew"] = movies["crew"].apply(lambda x: [i.replace(" ", "") for i in x])

Bunu yapmamızın sebebi:

Science Fiction → ScienceFiction

Sam Worthington → SamWorthington

Christopher Nolan → ChristopherNolan

şeklinde tek parça token elde etmek.

Bu recommendation kalitesini artırır.

**tags sütunu oluşturma**

Şimdi tüm önemli bilgileri tek sütunda birleştireceğiz.

In [40]:
movies["tags"] = (
    movies["overview"]
    + movies["genres"]
    + movies["keywords"]
    + movies["cast"]
    + movies["crew"]
)

Kontrol:

In [42]:
movies[["title", "tags"]].head()

,title,tags
0,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."


Artık her film için içerik kimliği oluşmuş oldu.

Örnek olarak bir film için tags içinde şunlar olabilir:

action,

adventure,

alien,

future,

SamWorthington,

JamesCameron

**Yeni dataframe oluşturma**

In [43]:
new_df = movies[["movie_id", "title", "tags"]].copy()

Burada sadece recommendation için gerekli olan 3 sütunu bırakıyoruz:

movie_id

title

tags

.copy() kullanmamız pandas warning almamak için daha güvenlidir.

**tags listesini düz metne çevirme**

In [44]:
new_df["tags"] = new_df["tags"].apply(lambda x: " ".join(x))
new_df["tags"] = new_df["tags"].apply(lambda x: x.lower())

Kontrol:

In [45]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


Önce:

["action", "adventure", "alien", "future"]

Sonra:

"action adventure alien future"

haline geliyor.

Tüm metni küçük harfe çevirmek de standardizasyon sağlar.

**Metinleri sayısal vektöre dönüştürme**

In [46]:
cv = CountVectorizer(max_features=5000, stop_words="english")
vectors = cv.fit_transform(new_df["tags"]).toarray()

Kontrol:

In [47]:
print("Vectors shape:", vectors.shape)

Vectors shape: (4806, 5000)


Burada her film artık sayısal bir vektörle temsil edilir.

satırlar → filmler

sütunlar → kelime özellikleri

max_features=5000 ile en önemli 5000 kelime seçilir.
stop_words="english" ile gereksiz İngilizce kelimeler çıkarılır.

**Cosine similarity hesaplama**

In [48]:
similarity = cosine_similarity(vectors)

Kontrol:

In [49]:
print("Similarity matrix shape:", similarity.shape)

Similarity matrix shape: (4806, 4806)


Bu matris her filmin diğer tüm filmlerle benzerlik skorunu tutar.

Örneğin:

similarity[0][1]

similarity[10][25]

iki film arasındaki benzerliktir.

Skor ne kadar yüksekse filmler o kadar benzerdir.

***Öneri Fonksiyonu Yazma***

In [50]:
def recommend(movie):
    movie = movie.lower()

    matching_movies = new_df[new_df["title"].str.lower() == movie]

    if matching_movies.empty:
        print("Film bulunamadı.")
        return

    index = matching_movies.index[0]
    distances = similarity[index]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )

    print(f"\n'{matching_movies.iloc[0]['title']}' için önerilen filmler:\n")

    for i in movie_list[1:6]:
        print(new_df.iloc[i[0]].title)

Bu fonksiyon:

Kullanıcının yazdığı filmi bulur

O filmin similarity skorlarını alır

En benzer filmleri sıralar

İlk 5 öneriyi gösterir

movie_list[1:6] kullanıyoruz çünkü ilk sırada filmin kendisi olur.

# **Sistemi Test Etme**

In [51]:
recommend("Avatar")


'Avatar' için önerilen filmler:

Titan A.E.
Small Soldiers
Independence Day
Ender's Game
Aliens vs Predator: Requiem


In [52]:
recommend("Inception")


'Inception' için önerilen filmler:

Duplex
The Helix... Loaded
Star Trek II: The Wrath of Khan
Timecop
Chicago Overcoat


In [53]:
recommend("The Dark Knight")


'The Dark Knight' için önerilen filmler:

The Dark Knight Rises
Batman Begins
Batman Returns
Batman Forever
Batman & Robin
